# Question 1 — corrected implementation
This notebook provides a cleaner implementation of SIR filtering, FFBSa smoothing, path-degeneracy diagnostics, posterior mean/variance plots, and the additive-functional variance experiment.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

SEED = 42
rng = np.random.default_rng(SEED)


In [ ]:

# -----------------------------
# Model and synthetic data
# -----------------------------
def state_mean(x_prev, t):
    return 0.5 * x_prev + 25.0 * x_prev / (1.0 + x_prev**2) + 8.0 * np.cos(1.2 * t)

def simulate_data(T, rng):
    x = np.empty(T + 1)
    y = np.empty(T + 1)
    x[0] = rng.normal(0.0, 1.0)
    y[0] = x[0]**2 / 20.0 + rng.normal(0.0, 1.0)
    for t in range(1, T + 1):
        x[t] = state_mean(x[t-1], t) + rng.normal(0.0, np.sqrt(0.1))
        y[t] = x[t]**2 / 20.0 + rng.normal(0.0, 1.0)
    return x, y

T = 200
x_true, y = simulate_data(T, rng)

plt.figure(figsize=(12, 5))
plt.plot(x_true, label="latent state")
plt.plot(y, label="observation", alpha=0.8)
plt.xlabel("time")
plt.legend()
plt.title("Synthetic data")
plt.show()


In [ ]:

# -----------------------------
# Standard SIR filter (bootstrap proposal)
# We keep BOTH:
#   - pre-resampling particles + normalized filtering weights
#   - resampled particles + ancestor indices
# -----------------------------
def obs_loglik(x, y_t):
    return -0.5 * (y_t - x**2 / 20.0)**2 - 0.5 * np.log(2.0 * np.pi)

def normalize_logweights(logw):
    m = np.max(logw)
    w = np.exp(logw - m)
    return w / np.sum(w)

def sir_filter(y, N, rng):
    T = len(y) - 1

    particles = np.empty((N, T + 1))            # pre-resampling particles X_t^{(i)}
    weights = np.empty((N, T + 1))              # normalized filtering weights W_t^{(i)}
    resampled_particles = np.empty((N, T + 1))  # equally weighted cloud after resampling
    ancestors = np.empty((N, T + 1), dtype=int) # parent of each resampled particle
    ess = np.empty(T + 1)

    # t = 0
    particles[:, 0] = rng.normal(0.0, 1.0, size=N)
    weights[:, 0] = normalize_logweights(obs_loglik(particles[:, 0], y[0]))
    ess[0] = 1.0 / np.sum(weights[:, 0]**2)
    idx0 = rng.choice(N, size=N, replace=True, p=weights[:, 0])
    resampled_particles[:, 0] = particles[idx0, 0]
    ancestors[:, 0] = idx0

    # t = 1, ..., T
    for t in range(1, T + 1):
        parent_idx = rng.integers(0, N, size=N)  # placeholder, overwritten below

        # propagate from equally weighted resampled cloud at time t-1
        prop = state_mean(resampled_particles[:, t-1], t) + rng.normal(0.0, np.sqrt(0.1), size=N)
        particles[:, t] = prop

        # since we resample every step, incremental weights are proportional to g(y_t | x_t)
        weights[:, t] = normalize_logweights(obs_loglik(particles[:, t], y[t]))
        ess[t] = 1.0 / np.sum(weights[:, t]**2)

        idx = rng.choice(N, size=N, replace=True, p=weights[:, t])
        resampled_particles[:, t] = particles[idx, t]
        ancestors[:, t] = idx

    return {
        "particles": particles,
        "weights": weights,
        "resampled_particles": resampled_particles,
        "ancestors": ancestors,
        "ess": ess,
    }


In [ ]:

# -----------------------------
# FFBSa using one forward filter output
# -----------------------------
def ffbsa_from_filter(filt, y, rng, M=1):
    particles = filt["particles"]
    weights = filt["weights"]
    N, TT = particles.shape

    paths = np.empty((M, TT))
    for m in range(M):
        j = rng.choice(N, p=weights[:, -1])
        paths[m, -1] = particles[j, -1]

        for t in range(TT - 1, 0, -1):
            x_next = paths[m, t]
            mean_prev = state_mean(particles[:, t-1], t)
            logbw = np.log(weights[:, t-1] + 1e-300) - 0.5 * (x_next - mean_prev)**2 / 0.1
            bw = normalize_logweights(logbw)
            j = rng.choice(N, p=bw)
            paths[m, t-1] = particles[j, t-1]
    return paths

def ffbsa(y, N, rng, M=1):
    filt = sir_filter(y, N, rng)
    return ffbsa_from_filter(filt, y, rng, M=M), filt


In [ ]:

# -----------------------------
# 1. KDE plots for filtering and smoothing at selected times
# -----------------------------
time_steps = [1, 50, 100, 150, 199]
N_values = [50, 500]

for n in time_steps:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=False, sharey=False)
    fig.suptitle(f"n = {n}", fontsize=14)

    for col, N in enumerate(N_values):
        local_rng = np.random.default_rng(SEED + 1000 * n + N)
        y_sub = y[:n+1]
        filt = sir_filter(y_sub, N, local_rng)
        smooth_paths = ffbsa_from_filter(filt, y_sub, local_rng, M=N)

        x_f = filt["particles"][:, -1]
        w_f = filt["weights"][:, -1]
        x_s = smooth_paths[:, -1]

        x_min = min(np.min(x_f), np.min(x_s)) - 1.0
        x_max = max(np.max(x_f), np.max(x_s)) + 1.0
        grid = np.linspace(x_min, x_max, 300)

        kde_f = gaussian_kde(x_f, weights=w_f)
        kde_s = gaussian_kde(x_s)

        axes[0, col].plot(grid, kde_f(grid))
        axes[0, col].fill_between(grid, kde_f(grid), alpha=0.3)
        axes[0, col].set_title(f"Filtering, N={N}")

        axes[1, col].plot(grid, kde_s(grid))
        axes[1, col].fill_between(grid, kde_s(grid), alpha=0.3)
        axes[1, col].set_title(f"Smoothing, N={N}")

    plt.tight_layout()
    plt.show()


In [ ]:

# -----------------------------
# 2. Path degeneracy: number of distinct ancestors over time
# -----------------------------
def distinct_ancestor_counts(filt):
    ancestors = filt["ancestors"]
    N, TT = ancestors.shape
    counts = np.empty(TT, dtype=int)

    alive = np.arange(N)
    counts[-1] = len(np.unique(alive))

    for t in range(TT - 1, 0, -1):
        alive = ancestors[alive, t]
        counts[t-1] = len(np.unique(alive))
    return counts

N = 50
filt_deg = sir_filter(y, N, np.random.default_rng(SEED + 7))
counts = distinct_ancestor_counts(filt_deg)

plt.figure(figsize=(10, 4))
plt.plot(np.arange(len(y)), counts, linewidth=2)
plt.xlabel("time")
plt.ylabel("number of distinct ancestors at final time")
plt.title("Path degeneracy diagnostic for SIR (N=50)")
plt.grid(alpha=0.3)
plt.show()

# Optional: smoothed trajectories for visual comparison
smooth_paths = ffbsa_from_filter(filt_deg, y, np.random.default_rng(SEED + 8), M=50)
plt.figure(figsize=(12, 5))
for i in range(50):
    plt.plot(smooth_paths[i], alpha=0.15, linewidth=0.8)
plt.xlabel("time")
plt.title("FFBSa backward-sampled trajectories (N=50)")
plt.grid(alpha=0.2)
plt.show()


In [ ]:

# -----------------------------
# 3. Filtering / smoothing posterior means
# -----------------------------
def filtering_mean(filt):
    return np.sum(filt["particles"] * filt["weights"], axis=0)

def filtering_var(filt):
    mu = filtering_mean(filt)
    return np.sum((filt["particles"] - mu[None, :])**2 * filt["weights"], axis=0)

def smoothing_mean_var(y, N, rng, M=200):
    filt = sir_filter(y, N, rng)
    paths = ffbsa_from_filter(filt, y, rng, M=M)
    return paths.mean(axis=0), paths.var(axis=0), filt

fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)
for row, N in enumerate([50, 500]):
    local_rng = np.random.default_rng(SEED + 20 + N)
    sm_mean, sm_var, filt = smoothing_mean_var(y, N, local_rng, M=200)
    fi_mean = filtering_mean(filt)

    axes[row].plot(x_true, label="true state", linewidth=2, color="black")
    axes[row].plot(fi_mean, label="filtering mean", linestyle=":", linewidth=1.8)
    axes[row].plot(sm_mean, label="smoothing mean", linestyle="--", linewidth=1.8)
    axes[row].set_title(f"Posterior means, N={N}")
    axes[row].grid(alpha=0.3)
    axes[row].legend()

plt.xlabel("time")
plt.tight_layout()
plt.show()


In [ ]:

# -----------------------------
# 4. Filtering / smoothing posterior variances
# -----------------------------
fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)
for row, N in enumerate([50, 500]):
    local_rng = np.random.default_rng(SEED + 40 + N)
    sm_mean, sm_var, filt = smoothing_mean_var(y, N, local_rng, M=200)
    fi_var = filtering_var(filt)

    axes[row].plot(fi_var, label="filtering variance", linestyle=":", linewidth=1.8)
    axes[row].plot(sm_var, label="smoothing variance", linestyle="--", linewidth=1.8)
    axes[row].set_title(f"Posterior variances, N={N}")
    axes[row].grid(alpha=0.3)
    axes[row].legend()

plt.xlabel("time")
plt.tight_layout()
plt.show()


In [ ]:

# -----------------------------
# 5. Additive functional experiments
# PF path-space estimator:
#   \hat S_n^{PF} = sum_i W_n^{(i)} sum_{k=0}^n X_{0:n}^{(i)}
# Here we build complete resampled genealogies and then use equal weights 1/N
# after resampling at time n.
# -----------------------------
def sir_with_genealogy(y, N, rng):
    T = len(y) - 1
    x_res = np.empty((N, T + 1))
    ancestors = np.empty((N, T + 1), dtype=int)

    # t=0
    x0 = rng.normal(0.0, 1.0, size=N)
    w0 = normalize_logweights(obs_loglik(x0, y[0]))
    a0 = rng.choice(N, size=N, replace=True, p=w0)
    x_res[:, 0] = x0[a0]
    ancestors[:, 0] = a0

    for t in range(1, T + 1):
        x_prop = state_mean(x_res[:, t-1], t) + rng.normal(0.0, np.sqrt(0.1), size=N)
        w = normalize_logweights(obs_loglik(x_prop, y[t]))
        a = rng.choice(N, size=N, replace=True, p=w)
        x_res[:, t] = x_prop[a]
        ancestors[:, t] = a

    return x_res, ancestors

def reconstruct_paths_from_genealogy(x_res, ancestors):
    N, TT = x_res.shape
    paths = np.empty((N, TT))
    idx = np.arange(N)
    paths[:, -1] = x_res[:, -1]
    curr = idx.copy()
    for t in range(TT - 1, 0, -1):
        curr = ancestors[curr, t]
        paths[:, t-1] = x_res[curr, t-1]
    return paths

def stheta_pf_runs(y, N, n, num_runs=50, seed=0):
    out = np.empty(num_runs)
    y_sub = y[:n+1]
    for r in range(num_runs):
        rrng = np.random.default_rng(seed + r)
        x_res, anc = sir_with_genealogy(y_sub, N, rrng)
        paths = reconstruct_paths_from_genealogy(x_res, anc)
        out[r] = np.mean(np.sum(paths[:, :n+1], axis=1))
    return out

def empirical_variance_pf(y, N, n, num_runs=50, seed=0):
    vals = stheta_pf_runs(y, N, n, num_runs=num_runs, seed=seed)
    return np.var(vals, ddof=1)


In [ ]:

# -----------------------------
# 6. FFBSa estimator variance experiment
# -----------------------------
def stheta_ffbsa_runs(y, N, n, num_runs=50, seed=0):
    out = np.empty(num_runs)
    y_sub = y[:n+1]
    for r in range(num_runs):
        rrng = np.random.default_rng(seed + 5000 + r)
        filt = sir_filter(y_sub, N, rrng)
        path = ffbsa_from_filter(filt, y_sub, rrng, M=1)[0]
        out[r] = np.sum(path[:n+1])
    return out

def empirical_variance_ffbsa(y, N, n, num_runs=50, seed=0):
    vals = stheta_ffbsa_runs(y, N, n, num_runs=num_runs, seed=seed)
    return np.var(vals, ddof=1)


In [ ]:

# -----------------------------
# 7. Plot rescaled empirical variances
# -----------------------------
N = 50
num_runs = 50
time_points = np.arange(1, 200)

var_pf = np.empty_like(time_points, dtype=float)
var_sm = np.empty_like(time_points, dtype=float)

for i, n in enumerate(time_points):
    var_pf[i] = empirical_variance_pf(y, N, n, num_runs=num_runs, seed=SEED + 100) / (n**2)
    var_sm[i] = empirical_variance_ffbsa(y, N, n, num_runs=num_runs, seed=SEED + 200) / n

plt.figure(figsize=(12, 5))
plt.plot(time_points, var_pf, linewidth=2, label=r"$\widehat{\mathrm{Var}}(\hat S_n^{PF})/n^2$")
tail_mean_pf = np.mean(var_pf[-50:])
plt.axhline(tail_mean_pf, linestyle="--", linewidth=1, label=f"tail mean = {tail_mean_pf:.3f}")
plt.xlabel("n")
plt.title("Rescaled empirical variance of PF path-space estimator")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(time_points, var_sm, linewidth=2, label=r"$\widehat{\mathrm{Var}}(\hat S_n^{SM})/n$")
tail_mean_sm = np.mean(var_sm[-50:])
plt.axhline(tail_mean_sm, linestyle="--", linewidth=1, label=f"tail mean = {tail_mean_sm:.3f}")
plt.xlabel("n")
plt.title("Rescaled empirical variance of FFBSa estimator")
plt.grid(alpha=0.3)
plt.legend()
plt.show()
